# Turn a Codebase into a Knowledge Graph

A runnable companion to the course project [*Turn a Codebase into a Knowledge Graph*](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/docs/projects/codebase-knowledge-graph). This notebook parses Python source files with the standard library's `ast` module, builds a directed graph of files/functions/classes/imports/calls with `networkx`, and visualizes and queries it.

**No API key, no signup, no network access needed after installing packages.** This is pure static analysis and graph theory — one of the easiest projects in the whole course to run in a hosted notebook, since there's nothing to configure and no secret to keep out of a shared notebook.

Works the same in Google Colab, Kaggle Notebooks, or Binder.


## Setup: install dependencies

These are the exact packages the local example project (`examples/codebase-knowledge-graph/pyproject.toml`) declares.

In [ ]:
!pip install -q networkx pyvis matplotlib


## Create a toy sample repo

The real companion example ships a small `sample_repo/` folder of toy files on disk. Since a fresh Colab/Kaggle/Binder session doesn't have that folder, we recreate the same three files here by writing them to disk directly — this keeps the notebook fully self-contained, with no `git clone` required.

The files: `utils.py` (two functions + a class), `models.py` (imports `utils`, defines `Order`), and `main.py` (imports both and ties them together) — with deliberate import/call relationships to explore.


In [ ]:
from pathlib import Path

SAMPLE_REPO = Path("sample_repo")
SAMPLE_REPO.mkdir(exist_ok=True)

(SAMPLE_REPO / "utils.py").write_text('''"""Small math/formatting helpers used by the rest of this sample repo."""


def add(a, b):
    """Add two numbers."""
    return a + b


def multiply(a, b):
    """Multiply two numbers."""
    return a * b


class Formatter:
    """Formats numbers as human-readable strings."""

    def format(self, value):
        """Formats `value` as a two-decimal string, e.g. 19.99."""
        return f"{value:.2f}"
''', encoding="utf-8")

(SAMPLE_REPO / "models.py").write_text('''"""Defines a simple Order model that leans on utils.py for its math."""

import utils


class Order:
    """Represents a customer order: a quantity of one item at a unit price."""

    def __init__(self, quantity, unit_price):
        self.quantity = quantity
        self.unit_price = unit_price

    def total(self):
        """Total price for this order, before tax."""
        return utils.multiply(self.quantity, self.unit_price)

    def total_with_tax(self, tax_rate):
        """Total price including tax, built on top of total()."""
        tax = utils.multiply(self.total(), tax_rate)
        return utils.add(self.total(), tax)
''', encoding="utf-8")

(SAMPLE_REPO / "main.py").write_text('''"""Entry point: builds a couple of sample orders and prints their totals."""

from models import Order
from utils import Formatter


def print_order_summary(order):
    """Prints a formatted one-line summary of an order's total."""
    formatter = Formatter()
    print(f"Total: {formatter.format(order.total())}")


def main():
    """Builds two sample orders and prints their summaries."""
    order1 = Order(3, 19.99)
    order2 = Order(1, 499.00)
    print_order_summary(order1)
    print_order_summary(order2)


if __name__ == "__main__":
    main()
''', encoding="utf-8")

print("Wrote sample_repo/ with:", sorted(p.name for p in SAMPLE_REPO.glob("*.py")))


## Step 1: Parse a single file's AST

`ast.parse` turns source code into the same tree structure the interpreter itself builds. `ast.walk` visits every node in that tree.


In [ ]:
import ast
from pathlib import Path

source = (SAMPLE_REPO / "utils.py").read_text(encoding="utf-8")
tree = ast.parse(source, filename="utils.py")

for node in ast.walk(tree):
    if isinstance(node, ast.FunctionDef):
        print("function:", node.name)
    elif isinstance(node, ast.ClassDef):
        print("class:", node.name)
    elif isinstance(node, ast.Import):
        for alias in node.names:
            print("import:", alias.name)
    elif isinstance(node, ast.ImportFrom):
        print("import from:", node.module)


## Step 2 & 3: Walk the whole repo, build the graph, add call edges

This mirrors `build_graph.py` from the companion example almost line-for-line: a `ParsedFile` dataclass holds what each file contributed, `parse_file` extracts it with `ast`, and `build_graph` does two passes over the repo — first adding every file/function/class/import as a node, then resolving `"calls"` edges by matching short names.


In [ ]:
from dataclasses import dataclass, field
import networkx as nx


@dataclass
class ParsedFile:
    """Everything extracted from one Python file's AST."""

    path: Path
    imports: list = field(default_factory=list)
    functions: list = field(default_factory=list)
    classes: dict = field(default_factory=dict)
    calls: dict = field(default_factory=dict)


def _called_names(func_node):
    """Best-effort list of names called inside a function/method body."""
    names = []
    for node in ast.walk(func_node):
        if isinstance(node, ast.Call):
            target = node.func
            if isinstance(target, ast.Name):
                names.append(target.id)
            elif isinstance(target, ast.Attribute):
                names.append(target.attr)
    return names


def parse_file(path):
    """Parses one Python file's AST; returns None and warns instead of crashing on a syntax error."""
    try:
        source = path.read_text(encoding="utf-8", errors="ignore")
        tree = ast.parse(source, filename=str(path))
    except SyntaxError as exc:
        print(f"Skipping {path}: syntax error ({exc.msg} at line {exc.lineno})")
        return None

    parsed = ParsedFile(path=path)

    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                parsed.imports.append(alias.name.split(".")[0])
        elif isinstance(node, ast.ImportFrom):
            if node.module:
                parsed.imports.append(node.module.split(".")[0])

    # Only tree.body -- top-level statements -- so a method nested in a
    # class isn't mistaken for a module-level function.
    for node in tree.body:
        if isinstance(node, ast.FunctionDef):
            parsed.functions.append(node.name)
            parsed.calls[node.name] = _called_names(node)
        elif isinstance(node, ast.ClassDef):
            methods = []
            for item in node.body:
                if isinstance(item, ast.FunctionDef):
                    methods.append(item.name)
                    qualified = f"{node.name}.{item.name}"
                    parsed.calls[qualified] = _called_names(item)
            parsed.classes[node.name] = methods

    return parsed


def build_graph(repo_path):
    """Walks every .py file under repo_path and builds a directed knowledge graph."""
    graph = nx.DiGraph()
    parsed_files = {}

    py_files = sorted(p for p in repo_path.rglob("*.py") if ".venv" not in p.parts)
    for path in py_files:
        parsed = parse_file(path)
        if parsed is None:
            continue
        parsed_files[path] = parsed

        rel = str(path.relative_to(repo_path))
        graph.add_node(rel, kind="file")

        for module in parsed.imports:
            graph.add_node(module, kind="module")
            graph.add_edge(rel, module, kind="imports")

        for func_name in parsed.functions:
            qualified = f"{rel}::{func_name}"
            graph.add_node(qualified, kind="function", short_name=func_name, file=rel)
            graph.add_edge(rel, qualified, kind="defines")

        for class_name, methods in parsed.classes.items():
            class_qualified = f"{rel}::{class_name}"
            graph.add_node(class_qualified, kind="class", short_name=class_name, file=rel)
            graph.add_edge(rel, class_qualified, kind="defines")
            for method_name in methods:
                method_qualified = f"{rel}::{class_name}.{method_name}"
                graph.add_node(method_qualified, kind="method", short_name=method_name, file=rel)
                graph.add_edge(class_qualified, method_qualified, kind="defines")

    # Second pass: resolve "calls" edges now that every def is a known node.
    by_short_name = {}
    for node, data in graph.nodes(data=True):
        if data.get("kind") in {"function", "method"}:
            by_short_name.setdefault(data["short_name"], []).append(node)

    for path, parsed in parsed_files.items():
        rel = str(path.relative_to(repo_path))
        for def_name, called_names in parsed.calls.items():
            caller = f"{rel}::{def_name}"
            if caller not in graph:
                continue
            for called_name in called_names:
                for target in by_short_name.get(called_name, []):
                    if target != caller:
                        graph.add_edge(caller, target, kind="calls")

    return graph


graph = build_graph(SAMPLE_REPO)
print(f"{graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")


## Step 4: Visualize the graph

Two options, both from `build_graph.py`:

- **`pyvis`** — an interactive, self-contained HTML page (drag, zoom, hover for details). Great for exploring.
- **`matplotlib`** — a static image. This is included as a reliable fallback, since `pyvis`'s interactive HTML doesn't always render cleanly inline on every hosted notebook platform (Kaggle and Binder in particular).

**In Google Colab specifically**, the `pyvis` HTML *can* be displayed inline with `IPython.display.HTML` — try the cell below. If it renders blank on your platform, skip straight to the matplotlib version underneath, which always works.


In [ ]:
_COLORS = {
    "file": "#3b82f6",      # blue
    "module": "#9ca3af",    # gray -- imported, but not a file we parsed ourselves
    "class": "#f59e0b",     # amber
    "function": "#10b981",  # green
    "method": "#10b981",
}

_EDGE_COLORS = {
    "defines": "#d1d5db",
    "imports": "#9ca3af",
    "calls": "#ef4444",
}


def visualize_pyvis(graph, output_path="graph.html"):
    """Renders the graph as a self-contained, interactive HTML file with pyvis."""
    from pyvis.network import Network

    net = Network(height="800px", width="100%", directed=True, notebook=False)
    net.barnes_hut()  # a physics layout that spaces nodes out instead of overlapping

    for node, data in graph.nodes(data=True):
        kind = data.get("kind", "module")
        label = data.get("short_name", node)
        net.add_node(node, label=label, title=f"{kind}: {node}", color=_COLORS.get(kind, "#9ca3af"))

    for source, target, data in graph.edges(data=True):
        kind = data.get("kind", "")
        net.add_edge(source, target, title=kind, color=_EDGE_COLORS.get(kind, "#d1d5db"))

    net.write_html(output_path)
    print(f"Wrote interactive graph to {output_path}")
    return output_path


html_path = visualize_pyvis(graph, "graph.html")


In [ ]:
# Colab: display the interactive pyvis HTML inline.
# (On Kaggle/Binder this cell may render blank or not at all -- that's expected,
# not a bug in the graph-building code above. Use the matplotlib cell below instead.)
from IPython.display import HTML

HTML(filename=html_path)


In [ ]:
# Reliable fallback that works everywhere: a static matplotlib image.
import matplotlib.pyplot as plt
import networkx as nx


def visualize_matplotlib(graph, output_path="graph.png"):
    """Renders a static PNG with matplotlib -- a decent fallback for a small repo."""
    fig, ax = plt.subplots(figsize=(12, 9))
    layout = nx.spring_layout(graph, seed=42, k=0.6)  # seed -> reproducible layout between runs

    node_colors = [_COLORS.get(graph.nodes[n].get("kind", "module"), "#9ca3af") for n in graph.nodes]
    labels = {n: graph.nodes[n].get("short_name", n) for n in graph.nodes}

    nx.draw_networkx_nodes(graph, layout, node_color=node_colors, node_size=500, ax=ax)
    nx.draw_networkx_labels(graph, layout, labels=labels, font_size=7, ax=ax)
    for kind, color in _EDGE_COLORS.items():
        edges = [(u, v) for u, v, d in graph.edges(data=True) if d.get("kind") == kind]
        nx.draw_networkx_edges(graph, layout, edgelist=edges, edge_color=color, ax=ax, arrows=True)

    ax.set_title("Codebase knowledge graph")
    ax.axis("off")
    fig.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.show()
    return output_path


visualize_matplotlib(graph, "graph.png")


## Step 5: Query the graph

A graph you can only look at is already useful; a graph you can *ask questions of* is more useful. Both queries below are plain `networkx` traversal over `out_edges`/`in_edges` -- no new machinery needed.


In [ ]:
def what_does_it_call(graph, short_name):
    """Every node whose short name matches `short_name`, and everything it calls."""
    results = []
    for node, data in graph.nodes(data=True):
        if data.get("short_name") == short_name or node == short_name:
            callees = [t for _, t, d in graph.out_edges(node, data=True) if d.get("kind") == "calls"]
            results.append((node, callees))
    return results


def who_imports(graph, module_name):
    """Every file node with an "imports" edge pointing at `module_name`."""
    if module_name not in graph:
        return []
    return [src for src, _, d in graph.in_edges(module_name, data=True) if d.get("kind") == "imports"]


In [ ]:
# Query: what does Order.total_with_tax call?
for node, callees in what_does_it_call(graph, "total_with_tax"):
    if callees:
        print(f"{node} calls: {', '.join(callees)}")
    else:
        print(f"{node} calls nothing this tool could resolve.")

print()

# Query: what imports utils?
importers = who_imports(graph, "utils")
print(f"'utils' is imported by: {', '.join(importers) if importers else '(nobody)'}")


## Next steps

- Point `build_graph` at a bigger, real repository instead of the toy `sample_repo/` -- e.g. `!git clone` a public GitHub project into this notebook and re-run the cells above against it.
- Add an "inherits from" edge kind by reading `ast.ClassDef.bases`.
- Try `nx.pagerank(graph)` or in-degree centrality to find the most "central" functions instead of eyeballing the visualization.
- See the full lesson (with a step-by-step walkthrough of every cell above) at [`docs/projects/codebase-knowledge-graph/index.md`](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/docs/projects/codebase-knowledge-graph) and the local, `uv`-based companion script at [`examples/codebase-knowledge-graph/build_graph.py`](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/examples/codebase-knowledge-graph).
